# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a reproducible workflow for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library (see the [Croissant specification](https://github.com/mlcommons/croissant)).

### Dataset Source
The dataset source is described using a Croissant schema and is accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(url)

metadata = dataset.metadata.to_json()

print(f"Dataset: {metadata['name']}")
print(f"DOI/Identifier: {metadata.get('identifier', 'N/A')}")
print(f"Description: {metadata['description']}")

## 2. Data Overview
Review available record sets, their fields, and IDs.

**Note:** In Croissant, each entity is uniquely referenced by its `@id`. All further references (record sets, fields, columns) will use the `@id` identifier.

In [ ]:
# List all record sets and their available fields, referencing by @id
record_sets = dataset.record_sets
if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    print("Record Sets (@id):")
    for rs_id, rs in record_sets.items():
        name = getattr(rs, 'name', '(no name)')
        print(f"  - {rs_id}  (name: {name})")
        if hasattr(rs, 'fields') and rs.fields:
            print("    Fields:")
            for field_id, field in rs.fields.items():
                field_name = getattr(field, 'name', '(no name)')
                print(f"      - {field_id}  (name: {field_name})")

### Example: View Records
Let's print out several records from a record set. (Pick a record set `@id` from the previous output.)

In [ ]:
# Get the list of available record set @ids
record_set_ids = list(dataset.record_sets.keys())
if not record_set_ids:
    print("No record sets available.")
else:
    # Pick the first record set @id for demonstration
    example_record_set_id = record_set_ids[0]
    print(f"Reading some sample records from record set: {example_record_set_id}\n")
    for i, record in enumerate(dataset.records(record_set=example_record_set_id)):
        if i >= 3:
            break
        print(record)

## 3. Data Extraction
Extract all data from the available record sets into Pandas DataFrames, referenced by their record set `@id`s.

- Use the record set and field `@id`s from the previous overview step.

In [ ]:
# Extract all record sets into DataFrames, keyed by their @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set {record_set_id}.")
        else:
            print(f"No records found for record set {record_set_id}.")
    except Exception as e:
        print(f"Failed to load records for record set {record_set_id}: {e}")
    
# Display available columns for the first DataFrame
if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns for {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We will illustrate how to process the tabular record set with common EDA steps: filtering numeric values, normalizing a field, and grouping/categorizing data.

In all steps, column and field references are made using their exact `@id` as shown in the data extraction step.

In [ ]:
# Example EDA: Filtering and normalizing a numeric field based on its @id
# We'll select the first DataFrame loaded (assume main tabular data)

import numpy as np

if not dataframes:
    print("No dataframes loaded.")
else:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")

    # Try to detect a likely numeric field @id and a group field @id
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        # Heuristic: look for columns that appear to be age, interval, or numeric
        if ('age' in col.lower() or 'interval' in col.lower() or 'years' in col.lower()) and df[col].dtype != 'O':
            numeric_field_id = col
            break
    if numeric_field_id is None:
        # Fallback: pick the first float/int column
        for col in df.select_dtypes(include=[np.number]).columns:
            numeric_field_id = col
            break
    # Heuristic: group by a variable like sex, anatomical_location, or similar
    for col in df.columns:
        if ('sex' in col.lower() or 'group' in col.lower() or 'location' in col.lower() or 'site' in col.lower()):
            group_field_id = col
            break

    if numeric_field_id is None:
        print("Could not detect a numeric field for EDA.")
    else:
        print(f"Numeric field @id selected: {numeric_field_id}")
        # Filter out entries with numeric field > threshold (e.g., >10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}: {len(filtered_df)} rows")

        # Normalize the numeric field
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nFirst 5 normalized entries:")
        print(filtered_df[[numeric_field_id, col_norm]].head())

        # Group by a field (if available)
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['mean', 'median', 'count'])
            print(f"\nGrouped statistics by {group_field_id}:")
            print(grouped_df)
        else:
            print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize distributions or relationships using column `@id`s.

- You may need to install matplotlib or seaborn (if not installed).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if EDA above succeeded
if 'filtered_df' in locals() and not filtered_df.empty and numeric_field_id is not None:
    # Histogram of the numeric field
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If group_field_id is available, show boxplot
    if group_field_id is not None and group_field_id in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()
else:
    print("No suitable numeric data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset using the Croissant standard and the `mlcroissant` library:

- Metadata, record sets, and fields can be programmatically discovered by their `@id`s.
- Tabular data record sets can be extracted and analyzed in Python, with references to exact field/column ids for reproducibility and interoperability.
- Simple EDA (filtering, normalization, grouping) and visualizations are possible using these programmatic references.

For further analysis, consult the [mlcroissant documentation](https://croissant.mlcommons.org/) and use the Croissant schema structure to identify more complex, reproducible data workflows.